# Week 6: the Fourier transform of images

**What we do.** We compute the 2D discrete Fourier transform of photographs, look at its magnitude and phase, test which of the two carries the structure of an image, show that the Fourier transform is not local, and compress an image by keeping only low frequencies (band coding).

**What to expect.** Most of the energy of a photograph is in low frequencies (the centre of the shifted spectrum). Surprisingly, the phase matters more than the magnitude. A tiny local change of the image changes almost every Fourier coefficient — the motivation for wavelets, which are local.

**How to read the code.** Functions come from `wavelets_lab.fourier` (transforms) and `wavelets_lab.images` (loading and display). Spectra are shown **centred**: the zero frequency is in the middle, the axes show frequency indices from about `-N/2` to `N/2`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import fourier, images

## Magnitude and phase of the 2D DFT

**What we do.** We transform the `camera` image and display the logarithm of the magnitude and the phase.

**What to expect.** The magnitude is largest in the centre (low frequencies: slow changes of brightness) and decreases outwards. The bright horizontal and vertical lines through the centre come from the jumps at the image borders (the DFT treats the image as periodic). The phase looks like noise to the eye.

**How to read the code.** `fourier.magnitude_phase(camera)` returns the centred `|F|` and `angle(F)`. `fourier.log_magnitude` computes `log(1 + |F|)`; without the logarithm only the central value would be visible. `show_image` labels the axes in pixels by default; for spectra we pass our own labels (the dictionary `labels`, unpacked with `**labels`) because the axes are now frequency indices. Index 256 is frequency 0.

In [ ]:
camera = images.load_test_image("camera")
magnitude, phase = fourier.magnitude_phase(camera)
labels = {"xlabel": "horizontal frequency index", "ylabel": "vertical frequency index"}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
images.show_image(camera, "image", ax=axes[0])
images.show_image(
    fourier.log_magnitude(magnitude), "log(1 + |F|)", ax=axes[1], scaled=True, **labels
)
images.show_image(phase, "phase angle(F)", ax=axes[2], scaled=True, **labels)
plt.tight_layout()
plt.show()

## How fast does the spectrum decay?

**What we do.** We plot the magnitude along the central row of the shifted spectrum (horizontal frequencies only).

**What to expect.** A peak at frequency 0 and a decay over several orders of magnitude towards high frequencies: most of the information is in few low-frequency coefficients. This is what makes transform compression possible.

**How to read the code.** After `fftshift`, frequency 0 is at index 256, so `np.arange(512) - 256` labels the frequencies. `semilogy` uses a logarithmic vertical axis.

In [ ]:
centre = magnitude.shape[0] // 2
frequencies = np.arange(magnitude.shape[1]) - magnitude.shape[1] // 2
plt.figure(figsize=(9, 3))
plt.semilogy(frequencies, magnitude[centre])
plt.xlabel("horizontal frequency index")
plt.ylabel("|F| (log scale)")
plt.title("central row of the spectrum")
plt.show()

## The phase carries the structure

**What we do.** We swap magnitudes and phases of two images (`camera`, reduced to 256×256, and `coins`) and rebuild images from the mixtures.

**What to expect.** Each mixed image looks like the image whose **phase** it uses: the phase tells where the edges are, the magnitude mostly how much energy each frequency has, which is similar for all photographs.

**How to read the code.** `camera[::2, ::2]` takes every second row and column (512 → 256). `fourier.from_magnitude_phase(magnitude, phase)` builds the spectrum `|F| e^{i·phase}` and inverts the DFT.

In [ ]:
camera_small = camera[::2, ::2]
coins = images.load_test_image("coins")[:256, :256]
mag_a, phase_a = fourier.magnitude_phase(camera_small)
mag_b, phase_b = fourier.magnitude_phase(coins)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
images.show_image(camera_small, "camera", ax=axes[0])
images.show_image(coins, "coins", ax=axes[1])
images.show_image(
    fourier.from_magnitude_phase(mag_b, phase_a),
    "|coins|, phase of camera",
    ax=axes[2],
    scaled=True,
)
images.show_image(
    fourier.from_magnitude_phase(mag_a, phase_b),
    "|camera|, phase of coins",
    ax=axes[3],
    scaled=True,
)
plt.tight_layout()
plt.show()

## Non-locality

**What we do.** We change a small 10×10 square of the image and compare the spectra of the original and the modified image.

**What to expect.** The change affects 0.04% of the pixels but almost 100% of the Fourier coefficients (the printed fraction is close to 1). The difference of the spectra is spread over the whole frequency plane. A Fourier coefficient "sees" the whole image; a wavelet coefficient sees only a small region.

**How to read the code.** `camera.copy()` makes an independent copy, so the original is not modified. `np.count_nonzero(difference > 1e-6) / difference.size` is the fraction of coefficients that changed.

In [ ]:
changed = camera.copy()
changed[100:110, 240:250] = 255.0
difference = np.abs(np.fft.fftshift(np.fft.fft2(changed) - np.fft.fft2(camera)))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
images.show_image(changed, "modified image (white square)", ax=axes[0])
images.show_image(
    fourier.log_magnitude(difference),
    "log(1 + |difference of spectra|)",
    ax=axes[1],
    scaled=True,
    **labels,
)
plt.tight_layout()
plt.show()
np.count_nonzero(difference > 1e-6) / difference.size

## Band coding with a square low-pass mask

**What we do.** We keep only the Fourier coefficients in a central square of half-width 16, 56 and 128 and reconstruct the image from them.

**What to expect.** Keeping 0.4% of the coefficients gives a very blurred image; 4.8% (the example of the lecture notes, compression ratio about 21, 0.38 bits per pixel) gives a reasonable image with "ringing" (oscillations) near edges — the 2D Gibbs phenomenon; 25% gives a good image with faint ringing. The titles show the kept fraction, the compression ratio, the bit rate and the PSNR.

**How to read the code.** `fourier.band_code(camera, half_width)` returns the reconstruction and the kept fraction; `fourier.compression_stats(kept)` converts the fraction to a ratio and bits per pixel. The PSNR is computed directly from its definition `10 log10(255² / MSE)`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, half_width in zip(axes, (16, 56, 128), strict=True):
    reconstruction, kept = fourier.band_code(camera, half_width)
    stats = fourier.compression_stats(kept)
    psnr = 10 * np.log10(255**2 / np.mean((reconstruction - camera) ** 2))
    title = f"kept {kept:.2%}, ratio {stats['ratio']:.1f}\n"
    title += f"{stats['bpp']:.2f} bpp, PSNR {psnr:.1f} dB"
    images.show_image(reconstruction, title, ax=ax)
plt.tight_layout()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Two modules are used: `wavelets_lab.images` (loading and display) and `wavelets_lab.fourier` (2D Fourier analysis and band coding). Functions marked *used internally* are called by the others.

**How to read the code.** Spectra are centred with `np.fft.fftshift` in all functions of this notebook.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.fft.fft2(image)` | 2D DFT (DFT of all rows, then of all columns) | 2D array → complex 2D array |
| `np.fft.fftshift(F)` | moves the zero frequency to the centre | array → array |
| `np.count_nonzero(condition)` | number of `True` entries | Boolean array → integer |
| `np.mean`, `np.log10` | mean value, decimal logarithm | array → number |
| `plt.semilogy(x, y)` | graph with logarithmic vertical axis | arrays → line |

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

### Module `wavelets_lab.fourier` (file `src/wavelets_lab/fourier.py`)

Fourier series, the discrete Fourier transform (DFT) and Fourier coding of images. The DFT convention is the one of `numpy.fft`:
`X[k] = sum_{n=0}^{N-1} x[n] exp(-2 pi i k n / N)`, inverse `x[n] = (1/N) sum_k X[k] exp(2 pi i k n / N)`, so own implementations can be compared with the library directly. Only NumPy (and `collections.abc.Callable` for type hints) is imported.

#### `fourier.magnitude_phase(image)`

**What it is.** The 2D DFT of an image split into magnitude `|F|` and phase `angle(F)`, both **centred**: the zero frequency (the mean intensity) is moved from the corner `[0, 0]` to the middle of the array with `fftshift`, so low frequencies are in the centre and high frequencies near the borders.

**Input.** `image` (array-like): greyscale image.

**Output.** Tuple `(magnitude, phase)` of `float64` arrays of the image's shape; phase values lie in `(-π, π]`.

**How to write it yourself.**
1. `spectrum = np.fft.fftshift(np.fft.fft2(np.asarray(image, dtype=np.float64)))`.
2. Return `np.abs(spectrum), np.angle(spectrum)`.

#### `fourier.log_magnitude(magnitude)`

**What it is.** Compresses a magnitude spectrum for display with `log(1 + |F|)`. The zero frequency is many orders of magnitude larger than the rest, so without the logarithm only one bright dot is visible.

**Input.** `magnitude` (array-like).

**Output.** `float64` array.

**How to write it yourself.** Return `np.log1p(np.asarray(magnitude, dtype=np.float64))` (`log1p(x) = log(1 + x)`, accurate also for small `x`).

#### `fourier.from_magnitude_phase(magnitude, phase)`

**What it is.** Rebuilds an image from a centred magnitude and phase (the inverse of `magnitude_phase`). Combining the magnitude of one image with the phase of another shows which of the two carries the structure.

**Input.** `magnitude`, `phase` (array-like, same shape, centred as from `magnitude_phase`).

**Output.** `float64` image: the real part of the inverse 2D DFT.

**How to write it yourself.**
1. Complex spectrum: `spectrum = magnitude * np.exp(1j * phase)`.
2. Undo the centring with `np.fft.ifftshift`, invert with `np.fft.ifft2`.
3. Return the real part (the imaginary part is only rounding noise for a valid spectrum, or is discarded for a mixed one).

#### `fourier.band_code(image, half_width)`

**What it is.** Band coding with a low-pass filter: all DFT coefficients outside a central square are set to zero and the image is reconstructed from the remaining ones. It is the simplest form of transform compression.

**Input.**
- `image` (array-like): greyscale image.
- `half_width` (`int`): half side of the kept square of frequencies.

**Output.** Tuple `(reconstruction, kept_fraction)`: the reconstructed `float64` image and the fraction of coefficients kept (e.g. `112**2 / 512**2 ≈ 0.048` for `half_width=56` and a 512×512 image).

**How to write it yourself.**
1. Centred spectrum: `spectrum = np.fft.fftshift(np.fft.fft2(image))`.
2. `mask = lowpass_mask(image.shape, half_width)`.
3. Zero the rest and invert: `np.fft.ifft2(np.fft.ifftshift(spectrum * mask)).real`.
4. Return the reconstruction and `float(mask.mean())` (the mean of a Boolean array is the fraction of `True`).

#### `fourier.lowpass_mask(shape, half_width)` (used internally)

**What it is.** A square low-pass mask for a centred spectrum: `True` on a `(2 * half_width) × (2 * half_width)` square around the centre (low frequencies), `False` elsewhere.

**Input.** `shape` (`tuple[int, int]`), `half_width` (`int`).

**Output.** Boolean array of shape `shape`.

**How to write it yourself.**
1. `mask = np.zeros(shape, dtype=bool)`; centre `(shape[0] // 2, shape[1] // 2)`.
2. Set `mask[c0 - half_width : c0 + half_width, c1 - half_width : c1 + half_width] = True`.

#### `fourier.compression_stats(kept_fraction, bits_per_pixel=8.0)`

**What it is.** Converts the kept fraction of coefficients into the compression ratio `1 / kept_fraction` and the bit rate `bits_per_pixel * kept_fraction` (bits per pixel, bpp), assuming each kept coefficient costs as much as one original pixel.

**Input.**
- `kept_fraction` (`float` in `(0, 1]`).
- `bits_per_pixel` (`float`, default 8): bits per stored coefficient.

**Output.** `dict` with keys `"ratio"` and `"bpp"`. Raises `ValueError` outside `(0, 1]`.

**How to write it yourself.**
1. Check `0 < kept_fraction <= 1`.
2. Return `{"ratio": 1 / kept_fraction, "bpp": bits_per_pixel * kept_fraction}`.